# Amazon ML Challenge 2026 — Business Entity Resolution · team **neural_nexus**

This notebook imports the dataset and the code (from GitHub), trains, predicts, and writes the submission
files. All logic lives in the repository (`scripts/kaggle_runner.py`), so code fixes arrive with the
`git clone` in step 1 — this notebook never needs re-importing.

### One-time setup (right-hand panel / top menu)
1. **Accelerator:** `GPU T4 x2` · **Internet:** `On`
2. **Input:** *Add Input → Your Datasets →* **`amazon-ml`**
3. **Hugging Face token (checkpoints):** *Add-ons → Secrets → Add Secret*; Label **`HF_TOKEN`**, Value: your
   token (the part after `HF_TOKEN=` in your `.env`; it needs **write** access). Tick the checkbox so the
   secret is attached to this notebook.

### Checkpoints and resuming
With the token, every finished stage is saved to a **private** Hugging Face repo
(`<your-hf-user>/amazon-ml-ber-work`). If a session stops (12 h limit, crash, closed tab), simply run the
notebook again: finished stages are restored and skipped, and the run continues where it stopped.
Set `FRESH_START = True` only when you want to throw the saved progress away (e.g. after changing the code).

### How to run
* **Rehearsal (~10–15 min):** `RUN_SMOKE_FIRST = True`, `RUN_FULL = False` → *Run All*. Must end with
  `SMOKE TEST PASSED`.
* **Full run:** `RUN_SMOKE_FIRST = False`, `RUN_FULL = True` → *Save Version → Save & Run All (Commit)*.
  Results are in the version's **Output** tab.

In [ ]:
# ---- settings -------------------------------------------------------------------------------------
REPO_URL = "https://github.com/Mveen3/amazon-ml-challange.git"
BRANCH = "main"
DATASET_SLUG = "amazon-ml"      # your Kaggle dataset name
TEAM_NAME = "neural_nexus"      # -> neural_nexus_submission.zip
RUN_SMOKE_FIRST = True          # True: rehearse the full run on a 5k-entity sample (incl. checkpoint round-trip)
RUN_FULL = True                 # False: stop after setup (+ rehearsal)
USE_CROSS_ENCODER = True        # False: skip the neural stage (saves ~1-1.5 h)
USE_HF_CHECKPOINT = True        # needs the HF_TOKEN Kaggle Secret (see above)
FRESH_START = False             # True: delete saved progress on Hugging Face and start from scratch
EXTRA_OVERRIDES = []            # any config value, e.g. ["ingest.train_s1_frac=0.6"]
CONFIG = "configs/track_g.yaml"
THEN_CONFIG = "configs/track_f.yaml"  # runs right after CONFIG in the same session ("" = none)

import importlib, subprocess, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "amazon-ml-challange"
PKG = REPO_DIR / "code" / "business_entity_resolution"

## 1. Import the code from GitHub

In [ ]:
def _git(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr).strip()[-3000:])
    r.check_returncode()

if (REPO_DIR / ".git").exists():
    _git(f"git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD")
else:
    _git(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
_git(f"git -C {REPO_DIR} log -1 --format='%h %s (%cd)'")

if str(PKG / "scripts") not in sys.path:
    sys.path.insert(0, str(PKG / "scripts"))
import kaggle_runner
importlib.reload(kaggle_runner)
R = kaggle_runner.KaggleRun(pkg=PKG, config=CONFIG, dataset_slug=DATASET_SLUG, team=TEAM_NAME,
                            use_cross_encoder=USE_CROSS_ENCODER, use_hf_checkpoint=USE_HF_CHECKPOINT,
                            fresh_start=FRESH_START, overrides=EXTRA_OVERRIDES)

## 2. Hardware check

In [ ]:
R.hardware()

## 3. Install the few packages Kaggle does not already have

In [ ]:
R.install()

## 4. Hugging Face token (from Kaggle Secrets; the value is never printed)

In [ ]:
R.secrets()

## 5. Import the dataset
Finds the 7 challenge files anywhere in the `amazon-ml` dataset (unpacking archives if needed).

In [ ]:
R.data()

## 6. Rehearsal (optional) — the full-run config on a 5k-entity sample
Includes the real cross-encoder on both GPUs and, with the token, a real checkpoint save → restore → skip
cycle on Hugging Face. Scores are far higher than on real data (random distractors); it only proves it all runs.

In [ ]:
if RUN_SMOKE_FIRST:
    R.smoke()

## 7. Full run
Each cell is one group of stages; the log shows progress and the session time used.
After a restart, finished stages print `already done` and are skipped.

In [ ]:
if RUN_FULL:
    R.stage("ingest,eda,mine,normalize")      # load TSVs, mine lookup tables, parse every record

In [ ]:
if RUN_FULL:
    R.stage("block")                          # candidate generation (TF-IDF kNN on GPU + exact keys)

In [ ]:
if RUN_FULL:
    R.stage("prerank,expand")                 # pre-ranker + floor tuned on the train ceiling -> candidates

In [ ]:
if RUN_FULL:
    R.stage("features")                       # round-1 pair features (CPU)

In [ ]:
if RUN_FULL:
    R.stage("r1")                             # round-1 GBDT (XGBoost on GPU), OOF on train

In [ ]:
if RUN_FULL:
    R.stage("ce_train,ce_infer")              # cross-encoder on the uncertain band (both T4s, fp16)

In [ ]:
if RUN_FULL:
    R.stage("r2,gate,tune,predict,outputs")   # round 2, entity gate, thresholds, submission files + validator

## 8. Results

In [ ]:
if RUN_FULL:
    R.results()

## 9. Final submission package (zip in the required layout + trained-models bundle)

In [ ]:
if RUN_FULL:
    R.package()

## 10. Next track in the same session (`THEN_CONFIG`)
Starts when the run above has finished. The run above's files move to `/kaggle/working/<its config name>/`;
the top-level files are the next track's.

In [ ]:
if RUN_FULL and THEN_CONFIG:
    R = R.then(THEN_CONFIG)
    R.full_run()

### Download (version → **Output** tab)
* `matching_results.tsv` — upload to the challenge portal
* `neural_nexus_submission.zip` — final package (fill in `Documentation_template.md` inside it before handing in)
* `neural_nexus_models.tar.gz` — trained models (inference-only reproduction)
* `reports/`, `logs/` — validation numbers and full logs